<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 多头注意力 + 数据加载（精简版）

本 notebook 是 ch03 的核心代码精华，将第 2 章的数据加载管道与第 3 章的多头注意力实现整合在一起。

```
完整流程:
  原始文本 → BPE 分词 → 滑动窗口 → token embedding + pos embedding
         → 多头因果注意力 → 上下文向量
```


In [1]:
# NBVAL_IGNORE_OUTPUT
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.2.2


完整的章节代码在 [ch03.ipynb](./ch03.ipynb) 中。

本 notebook 包含核心要点：多头注意力实现（加上第 2 章的数据加载管道）。


## 第 1 部分：数据加载管道（来自第 2 章）

```
GPTDatasetV1:
  原始文本 → BPE 编码 → 滑动窗口切片
  → input_chunk  (x):  token[i : i+max_length]
  → target_chunk (y):  token[i+1 : i+max_length+1]  ← 错位一格

create_dataloader:
  dataset → DataLoader(batch_size, shuffle)
  → 每次 yield 一个 batch: (x_batch, y_batch)
  shape: (batch_size, max_length)
```

**嵌入流程**：
```
token IDs → token_embedding → (batch, seq_len, output_dim)
                            +
位置索引 → pos_embedding   → (seq_len, output_dim)
                            =
input_embeddings            → (batch, seq_len, output_dim)
```


In [2]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    # GPT 数据集：将文本转换为输入-目标 token ID 对

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 用 BPE 分词器将整个文本编码为 token ID
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # 用滑动窗口将 token 序列切分为长度为 max_length 的重叠序列
        # stride 控制窗口滑动步长（stride=max_length 时无重叠）
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]           # 输入：当前窗口
            target_chunk = token_ids[i + 1: i + max_length + 1]  # 目标：右移一格
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # 创建数据加载器：文本 → 分词 → 数据集 → DataLoader
    tokenizer = tiktoken.get_encoding("gpt2")          # GPT-2 BPE 分词器
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader


# 读取示例文本
with open("small-text-sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

# 配置参数（模拟 GPT-2 small 的维度）
vocab_size = 50257      # GPT-2 词表大小
output_dim = 256        # 嵌入维度（GPT-2 small 为 768，这里缩小为 256）
max_len = 1024          # 最大上下文长度
context_length = max_len


# 创建 token 嵌入层和位置嵌入层
token_embedding_layer = nn.Embedding(vocab_size, output_dim)       # (50257, 256)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)  # (1024, 256)

# 用小窗口创建 DataLoader（max_length=4 方便演示）
max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)


In [3]:
# 遍历 DataLoader，取第一个 batch
for batch in dataloader:
    x, y = batch  # x: 输入 token IDs, y: 目标 token IDs

    # token IDs → token embeddings: (8, 4) → (8, 4, 256)
    token_embeddings = token_embedding_layer(x)

    # 位置索引 → 位置 embeddings: (4,) → (4, 256)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    # 最终输入 = token 嵌入 + 位置嵌入（广播：(8,4,256) + (4,256) → (8,4,256)）
    input_embeddings = token_embeddings + pos_embeddings

    break  # 只取第一个 batch


In [4]:
# 输入嵌入的 shape：(batch_size=8, seq_len=4, embed_dim=256)
print(input_embeddings.shape)


torch.Size([8, 4, 256])


# 第 2 部分：多头注意力实现（来自第 3 章）

提供两种实现：
- **Variant A**（简单版）：用 `nn.ModuleList` 堆叠多个单头注意力，直观但效率较低
- **Variant B**（高效版）：用矩阵 reshape + transpose 实现并行多头，效率高

```
Variant A: 逐头计算后拼接
  head_0(x) → ctx_0 (batch, seq, d_out)
  head_1(x) → ctx_1 (batch, seq, d_out)
  ...
  cat([ctx_0, ctx_1, ...], dim=-1) → (batch, seq, d_out * num_heads)
  out_proj → (batch, seq, d_out * num_heads)

Variant B: 矩阵并行
  x → Wq/Wk/Wv → Q/K/V (batch, seq, d_out)
  reshape → (batch, seq, num_heads, head_dim)
  transpose → (batch, num_heads, seq, head_dim)  ← 每个头并行
  attention → (batch, num_heads, seq, head_dim)
  transpose + reshape → (batch, seq, d_out)
  out_proj → (batch, seq, d_out)
```


## Variant A：简单实现（逐头计算后拼接）

**特点**：用 `nn.ModuleList` 创建 `num_heads` 个独立的 `CausalSelfAttention` 实例。

**优点**：直观易懂，每个头完全独立。
**缺点**：串行计算所有头，效率较低（`for head in self.heads`）。


In [5]:
class CausalSelfAttention(nn.Module):
    # 单头因果自注意力（带因果 mask 和 dropout）

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # 训练时随机置零，防止过拟合
        # register_buffer: 因果 mask 不参与梯度计算，但随模型一起移动到 GPU
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, n_tokens, d_in = x.shape  # b=batch_size, n_tokens=序列长度
        keys = self.W_key(x)          # (b, n_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 计算注意力分数: Q @ K^T → (b, n_tokens, n_tokens)
        attn_scores = queries @ keys.transpose(1, 2)
        # 因果 mask：将上三角（未来 token）置为 -inf
        attn_scores.masked_fill_(
            self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)
        # 缩放点积注意力：softmax(score / sqrt(d_k))
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和: weights @ V → (b, n_tokens, d_out)
        context_vec = attn_weights @ values
        return context_vec


class MultiHeadAttentionWrapper(nn.Module):
    # 多头注意力包装器：堆叠多个单头注意力后拼接

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # 创建 num_heads 个独立的单头注意力
        self.heads = nn.ModuleList(
            [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        # 输出投影：将拼接后的多头输出融合
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        # 逐头计算，然后在最后一维拼接: (b, seq, d_out) x num_heads → (b, seq, d_out*num_heads)
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)


In [6]:
torch.manual_seed(123)

context_length = max_length  # 4
d_in = output_dim            # 256

num_heads = 2
d_out = d_in // num_heads    # 128（每个头输出 128 维，2 头拼接 = 256 = output_dim）

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings      # (8, 4, 256)
context_vecs = mha(batch)     # 前向传播

# 输出 shape: (8, 4, 256) = (batch_size, seq_len, d_out x num_heads)
print("context_vecs.shape:", context_vecs.shape)


context_vecs.shape: torch.Size([8, 4, 256])


## Variant B：高效实现（矩阵并行）

**特点**：只用一组 Wq/Wk/Wv，通过 `view` + `transpose` 将输出隐式拆分到多个头。

**优点**：所有头并行计算，效率高（这是实际 GPT 使用的方法）。
**缺点**：维度变换较难理解。

**维度变换关键**：
```
x:           (batch, seq_len, d_in)
Wq(x):       (batch, seq_len, d_out)         ← d_out = num_heads * head_dim
.view(...):  (batch, seq_len, num_heads, head_dim)  ← 隐式拆分
.transpose:  (batch, num_heads, seq_len, head_dim)  ← 头维度提前，可并行
attn:        (batch, num_heads, seq_len, head_dim)
.transpose:  (batch, seq_len, num_heads, head_dim)
.view(...):  (batch, seq_len, d_out)         ← 拼接回来
out_proj:    (batch, seq_len, d_out)
```


In [7]:
class MultiHeadAttention(nn.Module):
    # 高效多头注意力：通过矩阵 reshape + transpose 实现并行多头计算

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # 每个头的维度，如 256/2 = 128

        # 只用一组 Wq/Wk/Wv（而非 num_heads 组），通过 reshape 隐式拆分
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 融合多头输出的线性层
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)       # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 通过 reshape 隐式拆分多头: (b, seq, d_out) -> (b, seq, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置: (b, seq, num_heads, head_dim) -> (b, num_heads, seq, head_dim)
        # 将 num_heads 维提到前面，这样矩阵乘法会并行计算所有头
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 缩放点积注意力（所有头并行）: Q @ K^T -> (b, h, seq, seq)
        attn_scores = queries @ keys.transpose(2, 3)

        # 因果 mask
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和 -> (b, h, seq, head_dim)
        # 转置回来: (b, h, seq, head_dim) -> (b, seq, h, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 拼接所有头: (b, seq, h, head_dim) -> (b, seq, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 最终线性投影

        return context_vec


In [8]:
torch.manual_seed(123)

context_length = max_length  # 4
d_in = output_dim            # 256
d_out = d_in                 # 256（与 Variant A 不同，这里 d_out 直接是 256）

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings      # (8, 4, 256)
context_vecs = mha(batch)     # 前向传播

# 输出 shape: (8, 4, 256) = (batch_size, seq_len, d_out)
print("context_vecs.shape:", context_vecs.shape)


context_vecs.shape: torch.Size([8, 4, 256])
